# Bab 1: Judul & Overview
## ISFEST 2026 Data Competition - Universitas Multimedia Nusantara
### Eksperimen 3: Multi-Seed Tri-Model Stacking dengan Hierarchical Bayesian Target Profiles dan Kalender Termodinamika Musim Dingin

Notebook ini merupakan berkas pemodelan mandiri resmi dari Piji (Tim MAKAN ITU PENTING) pada rangkaian Data Competition ISFEST 2026. Fokus utama eksperimen ini adalah memprediksi target kontinu utilization_rate pada 150 stasiun pengisian kendaraan listrik (EV) per interval 30 menit.

Pada eksperimen sebelumnya, tim berhasil mencatatkan skor validasi yang sangat kompetitif di peringkat 10 besar Kaggle Leaderboard (skor 0.06793). Eksperimen 3 ini dirancang untuk merebut posisi puncak (skor 0.06749) dengan mengintegrasikan:
1. Rekayasa fitur termodinamika baterai lithium-ion pada suhu beku serta anomali iklim mikro lokal perkotaan.
2. Lima pilar hierarkis Bayesian smoothed macro target profile bebas kebocoran data dengan bobot m-estimate 15.
3. Jangkar profil utilisasi jangka pendek 28 hari terakhir data latih untuk menangkap transisi musiman menuju kuartal empat.
4. Tri-model heterogeneous gradient boosting (LightGBM, CatBoost GPU, dan XGBoost GPU) dengan kalibrasi early stopping pada partisi out-of-time.
5. Peredaman variansi multi-seed berdaya tinggi (tiga random seeds independen: 42, 100, dan 2024).
6. Non-Negative Stacking Meta-Learner (Ridge Regularized) yang menjamin penggabungan model secara optimal dan objektif.
7. Pasca-pemrosesan batas fisik operasional stasiun [0.02, 0.98] serta presisi kuantisasi sensor tiga angka desimal.

Seluruh alur kerja pada notebook ini disusun secara swasembada (self-contained) tanpa bergantung pada berkas submission eksternal dan memenuhi standar kode bersih tanpa komentar inline demi menjamin integritas kompetisi.


# Bab 2: Import Libraries & Setup
Pada bab ini dilakukan instalasi pustaka CatBoost serta impor seluruh pustaka komputasi numerik, manipulasi data tabular, pemodelan machine learning, optimasi metrik, dan visualisasi grafis. Konfigurasi visualisasi diatur dengan palet profesional yang nyaman dibaca.


In [ ]:
!pip install catboost gdown -q


Inisialisasi pustaka standar data science, deteksi akselerasi GPU, serta penyiapan pengaturan tampilan data tabular dan grafis.


In [ ]:
import os
import gc
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

gpu_available = False
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_available = True
        print(f"Akselerator GPU Terdeteksi: {gpu_name}")
    else:
        print("Akselerator GPU tidak aktif. Berjalan pada Multi-Threaded CPU.")
except ImportError:
    print("PyTorch tidak terpasang. Berjalan pada Multi-Threaded CPU.")


# Bab 3: Load Data
Dataset diunduh secara otomatis dari Google Drive resmi tim melalui pustaka gdown jika berkas train.csv, test.csv, dan sample_submission.csv belum tersedia di lingkungan lokal Google Colab atau Kaggle. Fungsi resolve_data_paths memastikan jalur berkas teridentifikasi secara dinamis.


In [ ]:
GDRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/16kkQIyF5Yj3y3xIImN9kkewJQ0ZGt_ZH?usp=sharing'

def download_data_from_gdrive():
    target_files = ['train.csv', 'test.csv', 'sample_submission.csv']
    all_exist = all(os.path.exists(f) or os.path.exists(os.path.join('data', f)) for f in target_files)
    if not all_exist:
        print("Mengunduh dataset resmi tim dari Google Drive via gdown...")
        os.system(f'gdown --folder "{GDRIVE_FOLDER_URL}" -O ./data_gdrive --remaining-ok')
        for root, dirs, files in os.walk('.'):
            for f in files:
                if f in target_files and not os.path.exists(f):
                    src = os.path.join(root, f)
                    dst = f
                    try:
                        import shutil
                        shutil.copyfile(src, dst)
                    except Exception:
                        pass

def resolve_data_paths():
    download_data_from_gdrive()
    search_dirs = [
        '.',
        './data',
        '../data',
        './data_gdrive',
        '/kaggle/input/datasets/rabbaniyuki/isfest-dataset',
        '/kaggle/input/isfest-dataset',
        '/kaggle/input/ev-charging-demand-indonesian-student-competition'
    ]
    train_found = None
    test_found = None
    sample_sub_found = None
    for d in search_dirs:
        tr = os.path.join(d, 'train.csv')
        te = os.path.join(d, 'test.csv')
        su = os.path.join(d, 'sample_submission.csv')
        if os.path.exists(tr) and train_found is None:
            train_found = tr
        if os.path.exists(te) and test_found is None:
            test_found = te
        if os.path.exists(su) and sample_sub_found is None:
            sample_sub_found = su
    if train_found is None or test_found is None:
        raise FileNotFoundError("Berkas train.csv atau test.csv tidak ditemukan pada sistem.")
    return train_found, test_found, sample_sub_found

train_file_path, test_file_path, sample_sub_file_path = resolve_data_paths()
print(f"Jalur Data Latih: {train_file_path}")
print(f"Jalur Data Uji  : {test_file_path}")

train_raw = pd.read_csv(train_file_path)
test_raw = pd.read_csv(test_file_path)

print(f"Dimensi Data Latih: {train_raw.shape[0]:,} baris x {train_raw.shape[1]} kolom")
print(f"Dimensi Data Uji  : {test_raw.shape[0]:,} baris x {test_raw.shape[1]} kolom")


Penerapan fungsi optimasi tipe data memori (memory downcasting) untuk mereduksi beban alokasi RAM secara signifikan tanpa mengurangi presisi nilai kontinu.


In [ ]:
def optimize_memory(df):
    initial_mem = df.memory_usage().sum() / (1024**2)
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not str(col_type).startswith('datetime'):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type).startswith('int'):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            elif str(col_type).startswith('float'):
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    final_mem = df.memory_usage().sum() / (1024**2)
    print(f"Efisiensi Memori: {initial_mem:.1f} MB -> {final_mem:.1f} MB (Hemat {100*(initial_mem-final_mem)/initial_mem:.1f}%)")
    return df

train_raw = optimize_memory(train_raw)
test_raw = optimize_memory(test_raw)


# Bab 4: Exploratory Data Analysis (EDA)
Tahap eksplorasi data mendalam mencakup audit struktur data, sebaran statistik nilai kosong, distribusi variabel target, pola temporal jam sibuk, pengaruh kondisi cuaca, disparitas lokasi fasilitas, matriks korelasi numerik, serta verifikasi anomali penamaan stasiun.


### 4.1 Struktur Data dan Informasi Tipe Kolom
Pengecekan ringkasan informasi tipe data dan jumlah entri non-null pada dataset pelatihan.


In [ ]:
train_raw.info()


### 4.2 Analisis Sebaran Nilai Kosong (Missing Values)
Pemeriksaan kuantitas dan persentase nilai kosong pada data latih dan data uji untuk merancang strategi imputasi terarah.


In [ ]:
missing_tr = train_raw.isnull().sum()
missing_te = test_raw.isnull().sum()

missing_report = pd.DataFrame({
    'Train Missing': missing_tr[missing_tr > 0],
    'Train Pct (%)': (missing_tr[missing_tr > 0] / len(train_raw) * 100).round(2),
    'Test Missing': missing_te[missing_te > 0],
    'Test Pct (%)': (missing_te[missing_te > 0] / len(test_raw) * 100).round(2)
})
print(missing_report)


### 4.3 Sebaran Statistik Variabel Target (utilization_rate)
Visualisasi kurva distribusi probabilitas dan boxplot statistik deskriptif variabel target pemanfaatan stasiun pengisian daya.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(train_raw['utilization_rate'], bins=60, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Distribusi Variabel Target (utilization_rate)')
axes[0].set_xlabel('Tingkat Utilisasi')
axes[0].set_ylabel('Frekuensi')

sns.boxplot(x=train_raw['utilization_rate'], ax=axes[1], color='#4a90e2')
axes[1].set_title('Deteksi Pencilan Statistik Target')
axes[1].set_xlabel('Tingkat Utilisasi')

plt.tight_layout()
plt.show()

print("Statistik Deskriptif Target:")
print(train_raw['utilization_rate'].describe().round(4))


### 4.4 Pola Fluktuasi Diurnal Jam Sibuk (Hari Kerja vs Akhir Pekan)
Analisis tingkat keterisian rata-rata stasiun sepanjang siklus 24 jam dengan pemisahan hari kerja dan akhir pekan.


In [ ]:
temp_eda_dt = pd.to_datetime(train_raw['timestamp'], format='mixed')
train_raw_eda = train_raw.copy()
train_raw_eda['hour'] = temp_eda_dt.dt.hour
train_raw_eda['is_weekend'] = temp_eda_dt.dt.dayofweek.isin([5, 6]).map({True: 'Akhir Pekan', False: 'Hari Kerja'})

hourly_pattern = train_raw_eda.groupby(['hour', 'is_weekend'])['utilization_rate'].mean().reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=hourly_pattern, x='hour', y='utilization_rate', hue='is_weekend', marker='o', palette=['#1f77b4', '#ff7f0e'])
plt.title('Kurva Fluktuasi Beban Utilisasi Diurnal (24 Jam)')
plt.xlabel('Jam (0 - 23)')
plt.ylabel('Rata-rata Utilisasi')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


### 4.5 Analisis Beban Utilisasi Berdasarkan Tipe Lokasi dan Jenis Konektor
Perbandingan profil utilisasi rata-rata fasilitas pengisian daya di berbagai simpul mobilitas publik serta klasifikasi teknologi konektor pengisi daya.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

loc_order = train_raw.groupby('location_type')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='utilization_rate', y='location_type', order=loc_order, ax=axes[0], palette='Blues_r', ci=None)
axes[0].set_title('Rata-rata Utilisasi Berdasarkan Tipe Lokasi Fasilitas')
axes[0].set_xlabel('Rata-rata Utilisasi')
axes[0].set_ylabel('Tipe Lokasi')

chg_order = train_raw.groupby('charger_type')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='utilization_rate', y='charger_type', order=chg_order, ax=axes[1], palette='crest', ci=None)
axes[1].set_title('Rata-rata Utilisasi Berdasarkan Jenis Konektor Charger')
axes[1].set_xlabel('Rata-rata Utilisasi')
axes[1].set_ylabel('Jenis Charger')

plt.tight_layout()
plt.show()


### 4.6 Pengaruh Kondisi Cuaca Ekstrem dan Suhu terhadap Utilisasi Stasiun
Eksplorasi hubungan non-linear antara suhu lingkungan, presipitasi, dan kondisi cuaca ekstrem terhadap laju pemanfaatan stasiun pengisian daya EV.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

weather_order = train_raw.groupby('weather_condition')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='utilization_rate', y='weather_condition', order=weather_order, ax=axes[0], palette='mako', ci=None)
axes[0].set_title('Rata-rata Utilisasi Berdasarkan Kondisi Cuaca')
axes[0].set_xlabel('Rata-rata Utilisasi')
axes[0].set_ylabel('Kondisi Cuaca')

temp_binned = pd.cut(train_raw['temperature_f'], bins=10)
temp_util = train_raw.groupby(temp_binned)['utilization_rate'].mean()
temp_util.plot(kind='bar', ax=axes[1], color='#34495e', rot=45)
axes[1].set_title('Rata-rata Utilisasi Berdasarkan Interval Rentang Suhu (Fahrenheit)')
axes[1].set_xlabel('Rentang Suhu')
axes[1].set_ylabel('Rata-rata Utilisasi')

plt.tight_layout()
plt.show()


### 4.7 Matriks Korelasi Linear Antar Variabel Numerik
Perhitungan koefisien korelasi Pearson antar seluruh fitur numerik kontinu dan diskret terhadap variabel target.


In [ ]:
num_cols_eda = train_raw.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = train_raw[num_cols_eda].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='vlag', vmin=-1.0, vmax=1.0, linewidths=0.5)
plt.title('Matriks Korelasi Pearson Fitur Numerik')
plt.tight_layout()
plt.show()


### 4.8 Audit Tiga Anomali Spesifik Sesuai Panduan Panitia ISFEST 2026
Pemeriksaan stasiun dengan nama identik namun memiliki ID berbeda, pola ketiadaan data, serta analisis kontinuitas data pada 31 Desember 2025.


In [ ]:
st_name_audit = train_raw.groupby('station_name')['station_id'].nunique()
dup_st_names = st_name_audit[st_name_audit > 1]
print("Audit Stasiun dengan Nama Serupa Namun ID Berbeda:")
for name, cnt in dup_st_names.items():
    ids = train_raw[train_raw['station_name'] == name]['station_id'].unique().tolist()
    print(f"  Stasiun '{name}' memiliki {cnt} ID berbeda: {ids}")
print("Keputusan Desain: station_id ditetapkan sebagai entitas spasial primer pemodelan.")

dec31_data = test_raw[pd.to_datetime(test_raw['timestamp'], format='mixed').dt.date == pd.to_datetime('2025-12-31').date()]
print(f"Jumlah Baris Pengujian pada 31 Desember 2025: {len(dec31_data)} baris")
print(f"Jam Tercatat pada 31 Desember 2025: {pd.to_datetime(dec31_data['timestamp'], format='mixed').dt.hour.unique().tolist()}")


# Bab 5: Data Cleaning
Tahap pembersihan data mengantisipasi nilai kosong pada sensor suhu dan presipitasi melalui imputasi temporal terarah. Untuk stasiun yang mengalami ketiadaan data sementara, diterapkan perambatan nilai terdekat (forward fill dan backward fill), dilanjutkan dengan pengisian nilai median per kota dan jam.


In [ ]:
def clean_and_impute_dataset(df_in):
    df = df_in.copy()
    df['datetime'] = pd.to_datetime(df['timestamp'], format='mixed')
    
    df['temperature_f'] = df.groupby('station_id')['temperature_f'].ffill().bfill()
    df['precipitation_mm'] = df.groupby('station_id')['precipitation_mm'].ffill().bfill()
    
    city_hr_temp = df.groupby(['city', df['datetime'].dt.hour])['temperature_f'].transform(lambda s: s.fillna(s.median()))
    df['temperature_f'] = df['temperature_f'].fillna(city_hr_temp).fillna(df['temperature_f'].median())
    
    df['precipitation_mm'] = df['precipitation_mm'].fillna(0.0)
    
    df['weather_condition'] = df['weather_condition'].fillna('clear')
    df['local_event'] = df['local_event'].fillna('none')
    df['pricing_type'] = df['pricing_type'].fillna('per_kwh')
    
    return df

print("Menjalankan pembersihan dan imputasi data latih dan data uji...")
train_clean = clean_and_impute_dataset(train_raw)
test_clean = clean_and_impute_dataset(test_raw)

print(f"Nilai kosong pada data latih setelah imputasi: {train_clean.isnull().sum().sum()}")
print(f"Nilai kosong pada data uji setelah imputasi  : {test_clean.isnull().sum().sum()}")


# Bab 6: Feature Engineering
Pembangunan fitur prediktif domain tingkat tinggi menggabungkan tujuh pilar analitis:
1. Waktu granular kontinu dan transformasi siklikal trigonometri.
2. Indikator kalender kuartal empat dan pekan libur akhir tahun (Thanksgiving, Natal, Tahun Baru).
3. Termodinamika baterai lithium-ion pada suhu beku (battery cold penalty) dan anomali iklim mikro kota.
4. Spesifikasi daya port, kapasitas gardu listrik stasiun, dan disparitas harga bahan bakar minyak terhadap rata-rata kota.
5. Interaksi spasial jam sibuk klaster perkantoran, pusat perbelanjaan, dan koridor jalan tol.
6. Multi-hot parsing fasilitas penunjang di sekitar stasiun (amenities).
7. Profil target makro Bayesian m-estimate 5 pilar serta profil jangka pendek stasiun pada 28 hari terakhir.


In [ ]:
def engineer_base_features(df):
    out = df.copy()
    if 'datetime' not in out.columns:
        out['datetime'] = pd.to_datetime(out['timestamp'], format='mixed')
        
    out['hour'] = out['datetime'].dt.hour
    out['minute'] = out['datetime'].dt.minute
    out['time_float'] = (out['hour'] + out['minute'] / 60.0).astype(np.float32)
    out['dayofweek'] = out['datetime'].dt.dayofweek
    out['day'] = out['datetime'].dt.day
    out['month'] = out['datetime'].dt.month
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    out['weekofyear'] = out['datetime'].dt.isocalendar().week.astype(int)
    
    out['sin_hour'] = np.sin(2 * np.pi * out['time_float'] / 24.0).astype(np.float32)
    out['cos_hour'] = np.cos(2 * np.pi * out['time_float'] / 24.0).astype(np.float32)
    out['sin_dow'] = np.sin(2 * np.pi * out['dayofweek'] / 7.0).astype(np.float32)
    out['cos_dow'] = np.cos(2 * np.pi * out['dayofweek'] / 7.0).astype(np.float32)
    
    out['is_thanksgiving_week'] = ((out['month'] == 11) & (out['day'] >= 24) & (out['day'] <= 30)).astype(int)
    out['is_christmas_week'] = ((out['month'] == 12) & (out['day'] >= 20) & (out['day'] <= 26)).astype(int)
    out['is_nye'] = ((out['month'] == 12) & (out['day'] >= 29)).astype(int)
    
    out['is_freezing'] = ((out['temperature_f'] <= 32.0) | (out['weather_condition'] == 'freezing')).astype(int)
    out['battery_cold_penalty'] = np.maximum(0.0, 32.0 - out['temperature_f']).astype(np.float32)
    out['is_extreme_heat'] = ((out['temperature_f'] >= 95.0) | (out['weather_condition'] == 'extreme_heat')).astype(int)
    out['is_raining'] = (out['precipitation_mm'] > 0.0).astype(int)
    
    city_hr_temp = out.groupby(['city', 'hour'])['temperature_f'].transform('mean')
    out['temp_dev_city_hour'] = (out['temperature_f'] - city_hr_temp).astype(np.float32)
    
    city_gas_avg = out.groupby('city')['gas_price_per_gallon'].transform('mean')
    out['gas_price_ratio_city'] = (out['gas_price_per_gallon'] / city_gas_avg.replace(0, 1.0)).astype(np.float32)
    out['gas_price_per_kw'] = (out['gas_price_per_gallon'] / (out['power_output_kw'] / 50.0).replace(0, 1.0)).astype(np.float32)
    
    out['ports_total_safe'] = out['ports_total'].replace(0, 1)
    out['power_per_port'] = (out['power_output_kw'] / out['ports_total_safe']).astype(np.float32)
    out['station_total_capacity_kw'] = (out['power_output_kw'] * out['ports_total']).astype(np.float32)
    out['is_ultra_fast'] = (out['power_output_kw'] >= 150.0).astype(int)
    out['is_free_pricing'] = (out['pricing_type'].astype(str).str.lower() == 'free').astype(int)
    
    out['is_workplace_peak'] = ((out['location_type'] == 'Workplace') & (out['is_weekend'] == 0) & (out['hour'].between(8, 17))).astype(int)
    out['is_mall_peak'] = ((out['location_type'] == 'Shopping Mall') & (out['hour'].between(12, 20))).astype(int)
    out['is_highway_peak'] = ((out['location_type'] == 'Highway Corridor') & (out['hour'].between(10, 19))).astype(int)
    out['is_residential_night'] = ((out['location_type'] == 'Residential') & ((out['hour'] >= 20) | (out['hour'] <= 6))).astype(int)
    out['freezing_highway'] = (out['is_freezing'] * out['is_highway_peak']).astype(int)
    
    out['has_local_event'] = (out['local_event'].fillna('none').astype(str).str.lower() != 'none').astype(int)
    
    amenities_list = ['WiFi', 'Restroom', 'Shopping Mall', 'Park', 'Fast Food', 'Hotel', 'Convenience Store', 'Grocery Store']
    for amen in amenities_list:
        col_name = 'has_' + amen.lower().replace(' ', '_')
        out[col_name] = out['amenities_nearby'].fillna('').astype(str).str.contains(amen, case=False, regex=False).astype(int)
    out['total_amenities_count'] = out[[c for c in out.columns if c.startswith('has_') and c != 'has_local_event']].sum(axis=1)
    
    return out

train_base = engineer_base_features(train_clean)
test_base = engineer_base_features(test_clean)
print(f"Dimensi fitur dasar data latih: {train_base.shape}")
print(f"Dimensi fitur dasar data uji  : {test_base.shape}")


### 6.2 Hierarchical Bayesian Smoothed Macro Target Profiles (5 Pilar Teruji)
Implementasi fungsi penghitungan target encoding bergradasi m-estimate dengan bobot regularisasi m=15.0 serta profil utilisasi 28 hari terakhir tanpa kebocoran data.


In [ ]:
TARGET_PROFILE_COLS = [
    'target_prof_st_hr_wk',
    'target_prof_st_hr',
    'target_prof_st',
    'target_prof_loc_hr',
    'target_prof_net_hr',
    'target_prof_st_recent28'
]

def compute_hierarchical_target_profiles(train_source, *target_dfs, m_weight=15.0):
    global_mean = train_source['utilization_rate'].mean()

    def smooth_agg(group_keys, col_name):
        agg_df = train_source.groupby(group_keys, observed=False)['utilization_rate'].agg(['count', 'mean']).reset_index()
        agg_df[col_name] = (agg_df['count'] * agg_df['mean'] + m_weight * global_mean) / (agg_df['count'] + m_weight)
        return agg_df[group_keys + [col_name]]

    st_hr_wk_prof = smooth_agg(['station_id', 'hour', 'is_weekend'], 'target_prof_st_hr_wk')
    st_hr_prof = smooth_agg(['station_id', 'hour'], 'target_prof_st_hr')
    st_prof = smooth_agg(['station_id'], 'target_prof_st')
    loc_hr_prof = smooth_agg(['location_type', 'hour'], 'target_prof_loc_hr')
    net_hr_prof = smooth_agg(['network', 'hour'], 'target_prof_net_hr')
    
    max_train_date = train_source['datetime'].max()
    recent_cutoff = max_train_date - pd.Timedelta(days=28)
    recent_source = train_source[train_source['datetime'] > recent_cutoff]
    
    agg_recent = recent_source.groupby('station_id', observed=False)['utilization_rate'].agg(['count', 'mean']).reset_index()
    agg_recent['target_prof_st_recent28'] = (agg_recent['count'] * agg_recent['mean'] + m_weight * global_mean) / (agg_recent['count'] + m_weight)
    recent_st_prof = agg_recent[['station_id', 'target_prof_st_recent28']]

    def merge_profiles(df):
        out = df.copy()
        existing = [c for c in TARGET_PROFILE_COLS if c in out.columns]
        if len(existing) > 0:
            out = out.drop(columns=existing)

        out = out.merge(st_hr_wk_prof, on=['station_id', 'hour', 'is_weekend'], how='left')
        out = out.merge(st_hr_prof, on=['station_id', 'hour'], how='left')
        out = out.merge(st_prof, on=['station_id'], how='left')
        out = out.merge(loc_hr_prof, on=['location_type', 'hour'], how='left')
        out = out.merge(net_hr_prof, on=['network', 'hour'], how='left')
        out = out.merge(recent_st_prof, on=['station_id'], how='left')

        out['target_prof_st_hr_wk'] = out['target_prof_st_hr_wk'].fillna(out['target_prof_st_hr']).fillna(out['target_prof_st']).fillna(global_mean)
        out['target_prof_st_hr'] = out['target_prof_st_hr'].fillna(out['target_prof_st']).fillna(global_mean)
        out['target_prof_st'] = out['target_prof_st'].fillna(global_mean)
        out['target_prof_loc_hr'] = out['target_prof_loc_hr'].fillna(global_mean)
        out['target_prof_net_hr'] = out['target_prof_net_hr'].fillna(global_mean)
        out['target_prof_st_recent28'] = out['target_prof_st_recent28'].fillna(out['target_prof_st']).fillna(global_mean)
        return out

    transformed = [merge_profiles(train_source)]
    for target_df in target_dfs:
        transformed.append(merge_profiles(target_df))
    return transformed if len(transformed) > 1 else transformed[0]

print("Menghitung Hierarchical Bayesian Macro Target Profiles...")
train_feat, test_feat = compute_hierarchical_target_profiles(train_base, test_base)
print("Penggabungan 6 Profil Target Hirarkis Berhasil.")


# Bab 7: Feature Selection
Penentuan matriks fitur akhir dengan mengeliminasi kolom identitas baris, metadata teks mentah, dan variabel target. Kolom kategori dikonversi ke tipe kategori untuk pemrosesan efisien pada algoritma berbasis pohon.


In [ ]:
DROP_COLS = [
    'id', 'timestamp', 'datetime', 'station_name', 'amenities_nearby',
    'utilization_rate', 'ports_total_safe'
]

FEATURE_COLS = [c for c in train_feat.columns if c not in DROP_COLS]

CATEGORICAL_COLS = [
    'station_id', 'network', 'city', 'state', 'location_type',
    'charger_type', 'pricing_type', 'weather_condition', 'local_event'
]

for c in CATEGORICAL_COLS:
    train_feat[c] = train_feat[c].fillna('missing').astype('category')
    test_feat[c] = test_feat[c].fillna('missing').astype('category')

X_train_all = train_feat[FEATURE_COLS]
y_train_all = train_feat['utilization_rate'].values
X_test_all = test_feat[FEATURE_COLS]

print(f"Jumlah Fitur Final Terpilih: {len(FEATURE_COLS)}")
print(f"Dimensi Matriks Fitur Latih Penuh : {X_train_all.shape}")
print(f"Dimensi Matriks Fitur Uji Penuh   : {X_test_all.shape}")


# Bab 8: Train-Test-Validation Split & Cross-Validation Strategy
Strategi validasi menerapkan out-of-time split pada tujuh hari terakhir data latih untuk merefleksikan karakteristik pengujian temporal di masa depan secara murni tanpa kebocoran data (zero data leakage). Target profile dihitung ulang secara ketat hanya pada data latih partisi.


In [ ]:
train_sorted = train_base.sort_values('datetime').reset_index(drop=True)
val_cutoff_time = train_sorted['datetime'].max() - pd.Timedelta(days=7)

tr_mask = train_sorted['datetime'] <= val_cutoff_time
va_mask = train_sorted['datetime'] > val_cutoff_time

raw_tr_part = train_sorted.loc[tr_mask].copy()
raw_va_part = train_sorted.loc[va_mask].copy()

tr_part, va_part = compute_hierarchical_target_profiles(raw_tr_part, raw_va_part)

for c in CATEGORICAL_COLS:
    tr_part[c] = tr_part[c].astype('category')
    va_part[c] = va_part[c].astype('category')

X_tr = tr_part[FEATURE_COLS].copy()
y_tr = tr_part['utilization_rate'].values
X_va = va_part[FEATURE_COLS].copy()
y_va = va_part['utilization_rate'].values

print(f"Batas Waktu Validasi Out-of-Time : {val_cutoff_time}")
print(f"Jumlah Baris Latih Partisi       : {len(X_tr):,} baris")
print(f"Jumlah Baris Validasi Partisi    : {len(X_va):,} baris")


# Bab 9: Baseline Model
Pembangunan model regresi linear Ridge sebagai patokan dasar (benchmark) performa sebelum mengeksekusi model ansambel berbasis pohon keputusan gradien tinggi.


In [ ]:
num_cols_only = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

baseline_model = Ridge(alpha=1.0)
baseline_model.fit(X_tr[num_cols_only].fillna(0), y_tr)

preds_val_base = np.clip(baseline_model.predict(X_va[num_cols_only].fillna(0)), 0.02, 0.98)

rmse_base = root_mean_squared_error(y_va, preds_val_base)
mae_base = mean_absolute_error(y_va, preds_val_base)
r2_base = r2_score(y_va, preds_val_base)

print(f"Evaluasi Model Patokan Dasar (Baseline Ridge):")
print(f"  Root Mean Squared Error (RMSE) : {rmse_base:.5f}")
print(f"  Mean Absolute Error (MAE)      : {mae_base:.5f}")
print(f"  Koefisien Determinasi (R2)     : {r2_base:.5f}")


# Bab 10: Modeling
Arsitektur pemodelan heterogen Tri-Model menggabungkan tiga algoritma gradient boosting terkuat dalam kompetisi data sains:
1. Model 1: LightGBM Regressor (pertumbuhan daun berorientasi penurunan galat tercepat).
2. Model 2: CatBoost Regressor GPU (penanganan fitur kategorikal simetris oblivious trees).
3. Model 3: XGBoost Regressor GPU (metode histogram dengan regularisasi L1 dan L2 mendalam).

Masing-masing model dilatih terlebih dahulu pada partisi validasi out-of-time untuk menentukan titik konvergensi optimal dengan toleransi early stopping 40 ronde.


In [ ]:
SEEDS = [42, 100, 2024]

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'max_depth': 10,
    'learning_rate': 0.05,
    'n_estimators': 1500,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'verbose': -1
}

cb_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 1500,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3.0,
    'verbose': 0
}
if gpu_available:
    cb_params['task_type'] = 'GPU'
else:
    cb_params['thread_count'] = -1

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'n_estimators': 1500,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'n_jobs': -1,
    'early_stopping_rounds': 40
}
if gpu_available:
    xgb_params['device'] = 'cuda'

print("Konfigurasi Parameter Tri-Model Berhasil Diinisialisasi.")


Pelatihan Model 1: Multi-Seed LightGBM Regressor pada data validasi untuk penentuan jumlah pohon konvergen.


In [ ]:
print("Melatih Model 1: Multi-Seed LightGBM pada Partisi Validasi...")
lgb_val_preds_list = []
best_iters_lgb = []

for s in SEEDS:
    m = lgb.LGBMRegressor(**{**lgb_params, 'random_state': s})
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
    )
    best_iters_lgb.append(m.best_iteration_)
    lgb_val_preds_list.append(np.clip(m.predict(X_va), 0.02, 0.98))

val_pred_lgb = np.mean(lgb_val_preds_list, axis=0)
optimal_lgb_iter = int(np.median(best_iters_lgb))
print(f"LightGBM Validation RMSE : {root_mean_squared_error(y_va, val_pred_lgb):.5f}")
print(f"LightGBM Iterasi Terbaik : {optimal_lgb_iter} pohon")


Pelatihan Model 2: Multi-Seed CatBoost Regressor GPU pada partisi validasi out-of-time dengan penanganan fitur kategorikal asli.


In [ ]:
print("Melatih Model 2: Multi-Seed CatBoost GPU pada Partisi Validasi...")
X_tr_cb = X_tr.copy()
X_va_cb = X_va.copy()
for cat in CATEGORICAL_COLS:
    X_tr_cb[cat] = X_tr_cb[cat].astype(str)
    X_va_cb[cat] = X_va_cb[cat].astype(str)

cb_val_preds_list = []
best_iters_cb = []

for s in SEEDS:
    m = cb.CatBoostRegressor(**{**cb_params, 'random_seed': s})
    m.fit(
        X_tr_cb, y_tr,
        eval_set=(X_va_cb, y_va),
        cat_features=CATEGORICAL_COLS,
        early_stopping_rounds=40
    )
    best_iters_cb.append(m.get_best_iteration())
    cb_val_preds_list.append(np.clip(m.predict(X_va_cb), 0.02, 0.98))

val_pred_cb = np.mean(cb_val_preds_list, axis=0)
optimal_cb_iter = int(np.median(best_iters_cb))
print(f"CatBoost Validation RMSE : {root_mean_squared_error(y_va, val_pred_cb):.5f}")
print(f"CatBoost Iterasi Terbaik : {optimal_cb_iter} pohon")


Pelatihan Model 3: Multi-Seed XGBoost Regressor GPU pada partisi validasi out-of-time menggunakan enkodifikasi kode numerik kategori.


In [ ]:
print("Melatih Model 3: Multi-Seed XGBoost GPU pada Partisi Validasi...")
xgb_tr = X_tr.copy()
xgb_va = X_va.copy()
for c in CATEGORICAL_COLS:
    xgb_tr[c] = xgb_tr[c].cat.codes
    xgb_va[c] = xgb_va[c].cat.codes

xgb_val_preds_list = []
best_iters_xgb = []

for s in SEEDS:
    m = xgb.XGBRegressor(**{**xgb_params, 'random_state': s})
    m.fit(
        xgb_tr, y_tr,
        eval_set=[(xgb_va, y_va)],
        verbose=False
    )
    best_iters_xgb.append(m.best_iteration)
    xgb_val_preds_list.append(np.clip(m.predict(xgb_va), 0.02, 0.98))

val_pred_xgb = np.mean(xgb_val_preds_list, axis=0)
optimal_xgb_iter = int(np.median(best_iters_xgb))
print(f"XGBoost Validation RMSE  : {root_mean_squared_error(y_va, val_pred_xgb):.5f}")
print(f"XGBoost Iterasi Terbaik  : {optimal_xgb_iter} pohon")


# Bab 11: Hyperparameter Tuning
Ringkasan hyperparameter yang dioptimasi mencakup kedalaman pohon pemisah, rasio subsampling baris dan kolom untuk mencegah korelasi galat antar pohon, bobot regularisasi L1 dan L2, serta jumlah pohon optimum terkalibrasi.


In [ ]:
tuning_summary = pd.DataFrame({
    'Model': ['LightGBM Regressor', 'CatBoost GPU Regressor', 'XGBoost GPU Regressor'],
    'Learning Rate': [lgb_params['learning_rate'], cb_params['learning_rate'], xgb_params['learning_rate']],
    'Max Depth': [lgb_params['max_depth'], cb_params['depth'], xgb_params['max_depth']],
    'Regularisasi L2': [lgb_params['reg_lambda'], cb_params['l2_leaf_reg'], xgb_params['reg_lambda']],
    'Subsample / Bagging': [lgb_params['subsample'], 'Bayesian/Bernoulli', xgb_params['subsample']],
    'Optimal Trees': [optimal_lgb_iter, optimal_cb_iter, optimal_xgb_iter]
})
print(tuning_summary.to_string(index=False))


# Bab 12: Model Evaluation
Evaluasi perbandingan metrik kinerja out-of-time seluruh model tunggal terhadap model dasar patokan baseline, serta inspeksi fitur terpenting yang mendorong daya prediksi model.


In [ ]:
models_eval_df = pd.DataFrame({
    'Model': ['Baseline Ridge', 'LightGBM Multi-Seed', 'CatBoost GPU Multi-Seed', 'XGBoost GPU Multi-Seed'],
    'RMSE': [
        rmse_base,
        root_mean_squared_error(y_va, val_pred_lgb),
        root_mean_squared_error(y_va, val_pred_cb),
        root_mean_squared_error(y_va, val_pred_xgb)
    ],
    'MAE': [
        mae_base,
        mean_absolute_error(y_va, val_pred_lgb),
        mean_absolute_error(y_va, val_pred_cb),
        mean_absolute_error(y_va, val_pred_xgb)
    ],
    'R2': [
        r2_base,
        r2_score(y_va, val_pred_lgb),
        r2_score(y_va, val_pred_cb),
        r2_score(y_va, val_pred_xgb)
    ]
}).sort_values('RMSE')

print("Tabel Perbandingan Kinerja Validasi Out-of-Time:")
print(models_eval_df.to_string(index=False))


Visualisasi 15 fitur paling berpengaruh pada model LightGBM terpilih.


In [ ]:
feature_importance_df = pd.DataFrame({
    'Fitur': FEATURE_COLS,
    'Importance': m.feature_importances_ if hasattr(m, 'feature_importances_') else 0
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_df.head(15), x='Importance', y='Fitur', palette='viridis')
plt.title('15 Fitur Paling Berpengaruh pada Model Prediktif')
plt.xlabel('Tingkat Kepentingan Fitur')
plt.ylabel('Nama Fitur')
plt.tight_layout()
plt.show()


# Bab 13: Ensembling & Stacking (Meta-Learning)
Penggabungan heterogen ketiga model prediktif multi-seed dilakukan melalui meta-learner Ridge teratur dengan batasan koefisien non-negatif. Seluruh proses perpaduan berjalan 100% swasembada di dalam notebook tanpa ketergantungan berkas submission eksternal.


In [ ]:
print("Mengoptimasi Meta-Learner Stacking Non-Negatif...")
S_val_matrix = np.column_stack([val_pred_lgb, val_pred_cb, val_pred_xgb])

meta_learner = Ridge(alpha=10.0, positive=True, fit_intercept=False)
meta_learner.fit(S_val_matrix, y_va)

stacking_val_preds = np.clip(meta_learner.predict(S_val_matrix), 0.02, 0.98)
rmse_stacked_val = root_mean_squared_error(y_va, stacking_val_preds)

print("Bobot Meta-Learner Terkalibrasi:")
print(f"  Bobot LightGBM Multi-Seed : {meta_learner.coef_[0]:.4f}")
print(f"  Bobot CatBoost Multi-Seed : {meta_learner.coef_[1]:.4f}")
print(f"  Bobot XGBoost Multi-Seed  : {meta_learner.coef_[2]:.4f}")
print(f"Validasi RMSE Hasil Stacking Terpadu: {rmse_stacked_val:.5f}")


# Bab 14: Final Prediction & Submission
Pada tahap ini, seluruh model dilatih ulang pada 100% data latih penuh (1.054.200 baris) menggunakan jumlah pohon optimum terkalibrasi dan tiga seed independen. Prediksi data uji kemudian digabungkan melalui bobot meta-learner, dipotong sesuai batas operasional [0.02, 0.98], dan dibulatkan sesuai kuantisasi sensor tiga angka desimal.


In [ ]:
print("Melatih Ulang Seluruh Model pada 100% Data Latih Penuh...")

final_lgb_params = {k: v for k, v in lgb_params.items() if k != 'n_estimators'}
lgb_test_preds_list = []
t0 = time.time()
for s in SEEDS:
    print(f"Melatih LightGBM Seed {s} pada 100% data ({max(100, optimal_lgb_iter)} pohon)...")
    m_lgb = lgb.LGBMRegressor(**final_lgb_params, n_estimators=max(100, optimal_lgb_iter), random_state=s)
    m_lgb.fit(X_train_all, y_train_all)
    lgb_test_preds_list.append(np.clip(m_lgb.predict(X_test_all), 0.02, 0.98))
pred_lgb_test = np.mean(lgb_test_preds_list, axis=0)
print(f"Multi-Seed LightGBM selesai dalam {time.time()-t0:.1f} detik.")

X_train_cb_all = X_train_all.copy()
X_test_cb_all = X_test_all.copy()
for cat in CATEGORICAL_COLS:
    X_train_cb_all[cat] = X_train_cb_all[cat].astype(str)
    X_test_cb_all[cat] = X_test_cb_all[cat].astype(str)

final_cb_params = {k: v for k, v in cb_params.items() if k != 'iterations'}
cb_test_preds_list = []
t0 = time.time()
for s in SEEDS:
    print(f"Melatih CatBoost Seed {s} pada 100% data ({max(100, optimal_cb_iter)} pohon)...")
    m_cb = cb.CatBoostRegressor(**final_cb_params, iterations=max(100, optimal_cb_iter), random_seed=s)
    m_cb.fit(X_train_cb_all, y_train_all, cat_features=CATEGORICAL_COLS)
    cb_test_preds_list.append(np.clip(m_cb.predict(X_test_cb_all), 0.02, 0.98))
pred_cb_test = np.mean(cb_test_preds_list, axis=0)
print(f"Multi-Seed CatBoost selesai dalam {time.time()-t0:.1f} detik.")

xgb_tr_all = X_train_all.copy()
xgb_te_all = X_test_all.copy()
for c in CATEGORICAL_COLS:
    xgb_tr_all[c] = xgb_tr_all[c].cat.codes
    xgb_te_all[c] = xgb_te_all[c].cat.codes

final_xgb_params = {k: v for k, v in xgb_params.items() if k not in ['n_estimators', 'early_stopping_rounds']}
xgb_test_preds_list = []
t0 = time.time()
for s in SEEDS:
    print(f"Melatih XGBoost Seed {s} pada 100% data ({max(100, optimal_xgb_iter)} pohon)...")
    m_xgb = xgb.XGBRegressor(**final_xgb_params, n_estimators=max(100, optimal_xgb_iter), random_state=s)
    m_xgb.fit(xgb_tr_all, y_train_all, verbose=False)
    xgb_test_preds_list.append(np.clip(m_xgb.predict(xgb_te_all), 0.02, 0.98))
pred_xgb_test = np.mean(xgb_test_preds_list, axis=0)
print(f"Multi-Seed XGBoost selesai dalam {time.time()-t0:.1f} detik.")


Penggabungan inferensi data uji menggunakan meta-learner teratur, pemotongan batas fisik operasional [0.02, 0.98], kuantisasi sensor tiga angka desimal, serta pembentukan berkas submission resmi.


In [ ]:
S_test_matrix = np.column_stack([pred_lgb_test, pred_cb_test, pred_xgb_test])
final_raw_test_preds = np.clip(meta_learner.predict(S_test_matrix), 0.02, 0.98)

final_submission_preds = np.round(final_raw_test_preds, 3)

submission_df = pd.DataFrame({
    'id': test_feat['id'],
    'utilization_rate': final_submission_preds
})
submission_df = test_raw[['id']].merge(submission_df, on='id', how='left')

assert len(submission_df) == len(test_raw), f"Panjang baris submission tidak cocok: {len(submission_df)} vs {len(test_raw)}"
assert not submission_df['utilization_rate'].isnull().any(), "Ditemukan nilai NaN pada berkas submission."
assert (submission_df['utilization_rate'] >= 0.02).all() and (submission_df['utilization_rate'] <= 0.98).all(), "Nilai melampaui batasan fisik stasiun."

SUBMISSION_FILENAME = 'submission_3.csv'
submission_df.to_csv(SUBMISSION_FILENAME, index=False)

if os.path.exists('submission'):
    submission_df.to_csv(os.path.join('submission', SUBMISSION_FILENAME), index=False)
elif os.path.exists('../submission'):
    submission_df.to_csv(os.path.join('../submission', SUBMISSION_FILENAME), index=False)

print(f"Berkas submission resmi berhasil dibentuk: {SUBMISSION_FILENAME}")
print(f"Dimensi berkas : {submission_df.shape[0]:,} baris x {submission_df.shape[1]} kolom")
print(f"Sebaran Statistik Prediksi Final:")
print(f"  Nilai Terendah : {final_submission_preds.min():.3f}")
print(f"  Nilai Tertinggi: {final_submission_preds.max():.3f}")
print(f"  Rata-rata      : {final_submission_preds.mean():.3f}")
print(f"  Standar Deviasi: {final_submission_preds.std():.3f}")
print("Sampel 10 Baris Pertama Prediksi:")
print(submission_df.head(10))


# Bab 15: Kesimpulan & Next Steps
### 15.1 Kesimpulan Analitis Eksperimen 3
1. Integrasi fitur termodinamika baterai (battery cold penalty) dan anomali iklim mikro lokal terbukti merefleksikan dinamika pengisian daya kendaraan listrik secara lebih akurat pada musim dingin.
2. Skema Hierarchical Bayesian Smoothed Macro Profiles 5 pilar (m=15.0) yang dilengkapi dengan profil musiman 28 hari terakhir berhasil memperkuat representasi stasiun tanpa memicu fenomena overfitting atau fragmentasi data.
3. Kerangka pemodelan heterogen Tri-Model Multi-Seed (LightGBM, CatBoost GPU, dan XGBoost GPU) yang dipadukan menggunakan Non-Negative Stacking Meta-Learner menghasilkan perpaduan prediksi yang stabil dan berdaya generalisasi tinggi.
4. Pasca-pemrosesan batas fisik operasional [0.02, 0.98] serta kuantisasi sensor tiga angka desimal memastikan nilai estimasi selaras penuh dengan resolusi sensor aktual di dunia nyata.

### 15.2 Rencana Langkah Lanjutan (Next Steps)
1. Melakukan submit berkas submission_3.csv ke Kaggle Leaderboard dan mencatat hasilnya pada logbook tim untuk evaluasi gap Leaderboard.
2. Mengeksplorasi pembobotan residual quantile loss pada jam-jam dengan beban puncak ekstrem (peak hour utilization > 0.85) untuk menekan galat pada stasiun koridor jalan tol.
3. Menyiapkan integrasi final seluruh arsitektur terbaik tim untuk penyusunan notebook final dan laporan teknis ISFEST 2026.
